In [1]:
import numpy as np

# Load the data from the text file
Image_loaded = np.loadtxt("./Image.txt")

In [2]:
import h5py
import numpy as np

def load_ipole_unpol(fname):
    with h5py.File(fname, 'r') as file:
        # Read values
        dx = file["header/camera/dx"][()]
        dsource = file["header/dsource"][()]
        lunit = file["header/units/L_unit"][()]
        scale = file["header/scale"][()]
        fov_muas = dx / dsource * lunit * 2.06265e11
        
        # evpa_0 might not exist
        if "evpa_0" in file["header"]:
            evpa_0 = file["header/evpa_0"][()]
            # If it's bytes, decode to string
            if isinstance(evpa_0, bytes):
                evpa_0 = evpa_0.decode('utf-8')
        else:
            evpa_0 = "W"
        
        # Load and transpose unpol
        unpol = file["unpol"][()]
        unpol_t = unpol.T  # Transpose (equivalent to permutedims with (2,1))
        
        return unpol_t, fov_muas, scale, evpa_0

In [3]:
Image1, _, _, _= load_ipole_unpol("../../ipole/image.h5")

In [4]:
import numpy as np
import matplotlib.pyplot as plt

# --- Example setup ---
img1 = Image_loaded.T
img2 = Image1

# Compute absolute difference
diff_img = np.abs(img1 - img2)

# Field of view & axes setup
d_kpc = 16.9
d_cm = d_kpc * 3.086e21
fov_rg = 30
MBH = 6.2e9  # 4.063e6
GNEWT = 6.6742e-8 
CL = 2.99792458e10 
MSUN = 1.989e33 
L_unit = GNEWT * MBH * MSUN / (CL * CL)  # Length unit in gravitational radius (Rg)

half_fov_rg = fov_rg / 2
theta_rad = (half_fov_rg * L_unit) / d_cm
MUAS_PER_RAD = 2.06265e11
theta_μas = theta_rad * MUAS_PER_RAD

xlims = (-theta_μas, theta_μas)
ylims = (-theta_μas, theta_μas)

Ny, Nx = img1.shape
x = np.linspace(xlims[0], xlims[1], Nx)
y = np.linspace(ylims[0], ylims[1], Ny)

# --- Figure ---
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Common color scale for first two images
vmin = min(img1.min(), img2.min())
vmax = max(img1.max(), img2.max())
crange = (vmin, vmax)

# Panel 1: Image 1
ax1 = axes[0]
ax1.set_title("Jipole", fontsize=20)
ax1.set_xlabel("Relative R.A [μas]", fontsize=18)
ax1.set_ylabel("Relative Dec [μas]", fontsize=18)
ax1.set_xlim(xlims)
ax1.set_ylim(ylims)
ax1.set_aspect('equal')

im1 = ax1.imshow(img1, extent=[xlims[0], xlims[1], ylims[0], ylims[1]], 
                 origin='lower', cmap='hot', vmin=crange[0], vmax=crange[1])
cbar1 = plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
cbar1.set_label("Intensity", fontsize=14)

# Panel 2: Image 2
ax2 = axes[1]
ax2.set_title("Ipole", fontsize=20)
ax2.set_xlabel("Relative R.A [μas]", fontsize=18)
ax2.set_ylabel("Relative Dec [μas]", fontsize=18)
ax2.set_xlim(xlims)
ax2.set_ylim(ylims)
ax2.set_aspect('equal')

im2 = ax2.imshow(img2, extent=[xlims[0], xlims[1], ylims[0], ylims[1]], 
                 origin='lower', cmap='hot', vmin=crange[0], vmax=crange[1])
cbar2 = plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
cbar2.set_label("Intensity", fontsize=14)

# Panel 3: Absolute difference
ax3 = axes[2]
ax3.set_title("Absolute Difference", fontsize=20)
ax3.set_xlabel("Relative R.A [μas]", fontsize=18)
ax3.set_ylabel("Relative Dec [μas]", fontsize=18)
ax3.set_xlim(xlims)
ax3.set_ylim(ylims)
ax3.set_aspect('equal')

im3 = ax3.imshow(diff_img, extent=[xlims[0], xlims[1], ylims[0], ylims[1]], 
                 origin='lower', cmap='viridis')
cbar3 = plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)
cbar3.set_label("|delta Intensity|", fontsize=14)

plt.tight_layout()
plt.show()

ValueError: operands could not be broadcast together with shapes (160,160) (40,40) 

In [20]:
# Find the index of the maximum difference
max_idx = np.unravel_index(np.argmax(diff_img), diff_img.shape)
max_row, max_col = max_idx

# Get the maximum difference value
max_diff = diff_img[max_row, max_col]

# Get the corresponding coordinate values
x_coord = x[max_col]
y_coord = y[max_row]

print(f"Maximum difference: {max_diff}")
print(f"Pixel indices: row={max_row}, col={max_col}")
print(f"Coordinates: x={x_coord:.2f} μas, y={y_coord:.2f} μas")
print(f"img1 value at this pixel: {img1[max_row, max_col]}")
print(f"img2 value at this pixel: {img2[max_row, max_col]}")
print(f"error at {max_diff * 100/img1[max_row, max_col]}%")

# Optional: Mark this pixel on the plot
ax3.plot(x_coord, y_coord, 'r+', markersize=15, markeredgewidth=2)
plt.draw()

Maximum difference: 0.00014718565397948621
Pixel indices: row=12, col=20
Coordinates: x=1393.01 μas, y=-20895.13 μas
img1 value at this pixel: 0.003230389860258567
img2 value at this pixel: 0.0030832042062790806
error at 4.556281450428561%


<Figure size 432x288 with 0 Axes>

In [9]:
Image1[12,20]

0.0030832042062790806

In [10]:
Image_loaded[12,20]

0.0022925422377131936

In [3]:
import pyharm

/bin/bash: line 1: pyharm: command not found
